# Semantic search

Semantic search is:
- __Indexing__ (ingestion time)
    - __Chunking__ (when the docs are too big they must be split in chunks)
    - __Embedding__ (each chunk must be embedded by an embedding model)
- __Searching__ (query time)
    - __Similarity Search__ 
        - Once you have the index you can embed the user input with the same model and retrieve the indexed chunks whose embedding is most similar to the one of the user input
    - __Reranking__ [Optional]
        - When cosine similarity** is not good enough you can provide the top N chunks, by similarity, to a cross-encoder** (or to an LLM) to rerank them for better precision

Notes**:
- cosine similarity is the most commonly used similarity metric to compare embeddings  
- the cross-encoder is a model that takes the query and a single chunk candidate, then returns a relevance score for the couple. it's much heavier than the bi-encoder (the model used to embed the chunks and the query separately) because:
    - with bi-encoder, at run-time, you must run the model just once to generate the query embedding and then you compare this embedding with the indexed chunks' one.
    - with cross-encoder, at run-time, you run the model N times, once for each query + single_chunk_candidate.

Gotchas (hard-learned): 
- A general-purpose cross-encoder isn't automatically better than embeddings on a specific domain. (e.g.: ms-marco-MiniLM ranked Carbonara above Cacio e pepe for "cheese pasta", it's a worse result than cosine similarity).
- So you must evaluate reranked results against your golden dataset before trusting it, don't assume that reranking will be always an improvement.

## Indexing

### Chunking

Chunking a document is an AI Engineer task and it means to split the document in chunks before embedding each chunk. 

It's needed when a document to index:
- is __too big__ (technical limitation): the model couldn't be able to fit the whole document in the context
- covers __different topics__ (quality limitation): the resulting embedding would be a useless average of different topics 


There is no formal way to chunk documents, but here are some guidelines:

__Hard Upper Limit (Technical):__
- The hard upper-limit for a chunk is the context size of the embedder model (OpenAI usually sets the cap to 8192 tokens, Qwen goes from 32k to 128k, Gemini sets 2k tokens)
    - an input greater than this limit will be refused by the model
    - multiple chunks can be sent in batch to the model (even here there are limits, OpenAI usually sets the cap to 300k tokens) 
- The hard upper-limit is usually too big

__Best Practices (Quality):__
- 500-2k tokens is usually considered a good starting point but it changes according to the content
- The key is that mixing different topics in the same chunk makes the embedding a poor average representing none of the topics, so topics are usually separated in different chunks
    - chapters and paragraphs can be a hint of granularity
- Previous and next chunks usually have overlapping content (usually 10-15%) to avoid losing meaning when a sentence is split across two chunks

__Chunking techniques__ in real life:
- __Markdown documents__: split by structure (#, ##, etc.)
- __Recursive chunking__: tries a hierarchy of separators (\n\n, \n, sentences, chars) until every chunk is smaller than the chosen chunk size, then groups the chunks to reach the desired dimension 
- __LLM-based chunking__: rarely used because it's slow and expensive

Note: recursive chunking is content-agnostic, the first level (\n\n) still follows the author's paragraphs, but below that it splits blindly. You can't tell whether a chunk strategy will be good until you test it. 

__Chunking Quality__:
- Can only be measured on retrieval quality, via evals.
- Can be good for some queries and bad for others.



### Embedding

Here it is our dataset to index, containing recipes

Chunking won't be used because the content is small and each recipe covers just one topic

In [4]:
recipes = [
    {
        "title": "Carbonara",
        "text": "Carbonara. Serves 4. Ingredients: 320g spaghetti, 150g guanciale, 4 egg yolks, 100g pecorino romano, black pepper. Method: cut the guanciale into strips and fry it in a dry pan until crisp. Beat the yolks with grated pecorino and plenty of pepper. Cook the spaghetti al dente, drain and keep some pasta water. Toss off the heat with the guanciale and the egg mixture, loosening with pasta water until the sauce is smooth."
    },
    {
        "title": "Pesto alla genovese",
        "text": "Pesto alla genovese. Ingredients: 50g basil, 30g pine nuts, 2 garlic cloves, 60g parmesan, 20g pecorino, 100ml extra virgin olive oil, coarse salt. Method: crush garlic and salt in a mortar, add the basil and grind with a circular motion. Add the pine nuts until creamy, then the grated cheeses. Pour in the oil slowly while mixing. Never heat pesto: it is served raw."
    },
    {
        "title": "Tiramisu",
        "text": "Tiramisu. Serves 6. Ingredients: 500g mascarpone, 4 eggs, 100g sugar, 300g savoiardi biscuits, cold espresso, unsweetened cocoa. Method: whip the yolks with the sugar until pale, then fold in the mascarpone. Whip the egg whites to stiff peaks and fold them in gently. Dip the biscuits in the coffee without soaking them and layer with the cream. Chill for at least 4 hours and dust with cocoa before serving."
    },
    {
        "title": "Risotto alla milanese",
        "text": "Risotto alla milanese. Serves 4. Ingredients: 320g carnaroli rice, 1 sachet of saffron, 1 onion, 80g butter, 60g parmesan, beef stock, white wine. Method: soften the chopped onion in butter, toast the rice for two minutes. Deglaze with white wine. Add hot stock one ladle at a time while stirring. Halfway through, dissolve the saffron in a little stock and stir it in. Finish off the heat with cold butter and parmesan."
    },
    {
        "title": "Amatriciana",
        "text": "Amatriciana. Serves 4. Ingredients: 400g bucatini, 150g guanciale, 400g peeled tomatoes, pecorino romano, chilli, white wine. Method: fry the guanciale strips until crisp, deglaze with wine and set the meat aside. Add the crushed tomatoes and chilli, cook for 15 minutes. Return the guanciale, toss with the drained bucatini and finish with grated pecorino."
    },
    {
        "title": "Panna cotta",
        "text": "Panna cotta. Serves 6. Ingredients: 500ml double cream, 100g sugar, 8g gelatine leaves, 1 vanilla pod. Method: soak the gelatine in cold water for 10 minutes. Heat the cream with the sugar and vanilla seeds without boiling. Squeeze the gelatine and dissolve it in the hot cream. Pour into moulds and set in the fridge for at least 5 hours."
    },
    {
        "title": "Pasta e fagioli",
        "text": "Pasta e fagioli. Serves 4. Ingredients: 250g borlotti beans, 200g mixed pasta, 1 celery stick, 1 carrot, 1 onion, 2 tomatoes, rosemary, olive oil. Method: soak the beans overnight. Fry the chopped celery, carrot and onion, add the chopped tomatoes and the drained beans. Cover with water and simmer for an hour. Blend half the beans to thicken, then add the pasta and cook it directly in the soup."
    },
    {
        "title": "Cacio e pepe",
        "text": "Cacio e pepe. Serves 4. Ingredients: 320g tonnarelli, 200g pecorino romano, black peppercorns. Method: toast the coarsely crushed pepper in a pan. Cook the pasta in little water so it becomes very starchy. Melt the grated pecorino with warm pasta water into a smooth cream, keeping it off the heat or it will split. Toss the pasta with the pepper and finish with the pecorino cream."
    },
]

First of all we have to index it (we'll embed and add the couple recipe + embedding to an array)

In [5]:
from dataclasses import dataclass
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

@dataclass
class Embedded_recipe:
    title: str
    text: str
    embedding: list

chunks_index = []

client = OpenAI()

for recipe in recipes:
    print(f"Embedding recipe: {recipe['title']} - Text: {recipe['text'][:50]}...")  # Print first 50 characters of text for brevity

    response = client.embeddings.create(
        input=f"Title: {recipe['title']} --- Text: {recipe['text']}",
        model="text-embedding-3-small"
    )

    chunks_index.append(Embedded_recipe(
        title=recipe['title'],
        text=recipe['text'],
        embedding=response.data[0].embedding
    ))
    print(f"Usage: {response.usage} - number of embeddings: {len(response.data)} - Embedding length: {len(response.data[0].embedding)}")
    print(f"Embedding: {response.data[0].embedding[:5]}...\n")  # Print first 5 dimensions for brevity

Embedding recipe: Carbonara - Text: Carbonara. Serves 4. Ingredients: 320g spaghetti, ...
Usage: Usage(prompt_tokens=120, total_tokens=120) - number of embeddings: 1 - Embedding length: 1536
Embedding: [0.005893707275390625, 0.00513458251953125, 0.00516510009765625, -0.03021240234375, 0.0092926025390625]...

Embedding recipe: Pesto alla genovese - Text: Pesto alla genovese. Ingredients: 50g basil, 30g p...
Usage: Usage(prompt_tokens=111, total_tokens=111) - number of embeddings: 1 - Embedding length: 1536
Embedding: [-0.0276336669921875, 0.01371002197265625, 0.033966064453125, -0.0304412841796875, -0.032562255859375]...

Embedding recipe: Tiramisu - Text: Tiramisu. Serves 6. Ingredients: 500g mascarpone, ...
Usage: Usage(prompt_tokens=113, total_tokens=113) - number of embeddings: 1 - Embedding length: 1536
Embedding: [0.0121917724609375, -0.0259246826171875, 0.01824951171875, -0.073974609375, 0.0291290283203125]...

Embedding recipe: Risotto alla milanese - Text: Risotto alla milanese

Let's print the embedded recipes

In [6]:
for embedded_recipe in chunks_index:
    print(f"Embedded Recipe: {embedded_recipe.title} | Text: {embedded_recipe.text[:50]}... | Embedding length: {len(embedded_recipe.embedding)}")

Embedded Recipe: Carbonara | Text: Carbonara. Serves 4. Ingredients: 320g spaghetti, ... | Embedding length: 1536
Embedded Recipe: Pesto alla genovese | Text: Pesto alla genovese. Ingredients: 50g basil, 30g p... | Embedding length: 1536
Embedded Recipe: Tiramisu | Text: Tiramisu. Serves 6. Ingredients: 500g mascarpone, ... | Embedding length: 1536
Embedded Recipe: Risotto alla milanese | Text: Risotto alla milanese. Serves 4. Ingredients: 320g... | Embedding length: 1536
Embedded Recipe: Amatriciana | Text: Amatriciana. Serves 4. Ingredients: 400g bucatini,... | Embedding length: 1536
Embedded Recipe: Panna cotta | Text: Panna cotta. Serves 6. Ingredients: 500ml double c... | Embedding length: 1536
Embedded Recipe: Pasta e fagioli | Text: Pasta e fagioli. Serves 4. Ingredients: 250g borlo... | Embedding length: 1536
Embedded Recipe: Cacio e pepe | Text: Cacio e pepe. Serves 4. Ingredients: 320g tonnarel... | Embedding length: 1536


## Searching

### Semantic search

Now we will do a semantic search on the embedded recipes. 

We will define:
- a custom cosine similarity function 
- a search function to 
    - Compute the query embedding
    - Compute cosine similarity between the query embedding and each recipe embedding
    - Return the top N most related recipes

We will run the search.

In [7]:
def cosine_similarity(vec_a: list, vec_b: list) -> float:
    """Compute the cosine similarity between two vectors."""
    dot_product = sum(a * b for a, b in zip(vec_a, vec_b))
    norm_a = sum(a * a for a in vec_a) ** 0.5
    norm_b = sum(b * b for b in vec_b) ** 0.5
    return dot_product / (norm_a * norm_b) if norm_a and norm_b else 0.0

def search_recipes(query: str, chunks_index: list, top_k: int) -> list:
    """Search for recipes based on a query using cosine similarity."""
    query_embedding = client.embeddings.create(
        input=query,
        model="text-embedding-3-small"
    ).data[0].embedding

    # Calculate similarity scores
    similarities = [
        (embedded_recipe, cosine_similarity(query_embedding, embedded_recipe.embedding))
        for embedded_recipe in chunks_index
    ]

    # Sort by similarity score in descending order and return top_k results
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_k]


## execute the semantic search
query = "I would like to make a cheese pasta"
results = search_recipes(query, chunks_index, top_k=3)

print(f"Search results for query: '{query}'")
for embedded_recipe, score in results:
    print(f"Recipe: {embedded_recipe.title} | Similarity Score: {score:.4f}")

Search results for query: 'I would like to make a cheese pasta'
Recipe: Cacio e pepe | Similarity Score: 0.4660
Recipe: Carbonara | Similarity Score: 0.4550
Recipe: Pesto alla genovese | Similarity Score: 0.4120


### Reranking

To apply reranking on the top-n results retrieved by semantic search you could use:
- cross-encoding using cohere (a paid api)
- cross-encoding on-premise using sentence_transformers (downloads the model on premise)

Note:
- LLM as a judge (give question and answer to an LLM and ask to give a relatedness score between 1 and 5) can technically rerank but it's slow/costly for real-time retrieval, it's used in offline evals (see RAGAS)

In [22]:
# > poetry add sentence_transformers
from sentence_transformers import CrossEncoder
chunk_texts = [r[0].text for r in results] # usually you will rerank just a subset of the results retrieved by semantic search (top 50s)
                                           # note that you must pass only the clear text of query and chunks to the cross-encoder  

model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2") # the first time it will download the model (~100mbs)
reranked_results = model.rank(query, chunk_texts)

for r in reranked_results:
    corpus_id = r["corpus_id"] # the cross encoder returns just id and score for each document given in input, not the text given in input
    corpus_title = results[corpus_id][0].title # retrieve the recipe title using the corpus id
    print(f"reranked_result: {r} - corpus_title: {corpus_title}")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3413.62it/s]


reranked_result: {'corpus_id': 1, 'score': -8.236567497253418} - corpus_title: Carbonara
reranked_result: {'corpus_id': 0, 'score': -8.343669891357422} - corpus_title: Cacio e pepe
reranked_result: {'corpus_id': 2, 'score': -8.653650283813477} - corpus_title: Pesto alla genovese


Pay Attention! 

In this example the reranking result is worse than the original result! 

So always evaluate if reranking is improving or worsening your result by running it against your golden dataset. 